# 05 — Interpretabilidade dos Modelos (SHAP)

Analisa **quais features mais influenciam as previsões de inadimplência** de cada modelo usando SHAP.

Para **regressão**, os valores SHAP representam o impacto de cada feature na taxa de inadimplência prevista.

Modelos analisados: **Random Forest**, **XGBoost**, **Regressão Linear**.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib
from pathlib import Path
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
try:
    shap.initjs()          # so necessario para graficos JS interativos (nao usados aqui)
except Exception as e:
    print(f"shap.initjs() indisponivel ({type(e).__name__}) - segue com os graficos matplotlib")

BASE = Path().resolve().parent
PROC = BASE / "data" / "processed"
MDIR = PROC / "models"

with open(PROC / "ml_metadata.json", encoding="utf-8") as f:
    meta = json.load(f)

FEATURE_COLS = meta["feature_cols"]
TARGET_COL   = meta["target_col"]
SPLIT_DATE   = pd.Timestamp(meta["split_date"])

# Nomes amigáveis — gerados a partir de FEATURE_COLS para garantir consistência.
# Revisao A1: com defasagem de publicacao L, "inadim_lagk" e a taxa em t-L-k (serie PUBLICADA).
# Revisao A4c: "spread_pj" e o spread PJ de fato quando o esquema macro e o revisado.
L = int(meta.get("lag_scr", 0))
A4C = meta.get("macro_esquema") == "a4c"
_name_map = {
    **{f"inadim_lag{k}": f"Inadim. t-{k+L}" for k in (1, 2, 3, 6, 12)},
    "inadim_roll3_mean": "Media 3m", "inadim_roll6_mean": "Media 6m",
    "inadim_roll12_mean": "Media 12m",
    "inadim_roll3_std": "Desvio 3m", "inadim_roll6_std": "Desvio 6m",
    "inadim_delta1": "Delta 1m", "inadim_delta3": "Delta 3m",
    "vencido_lag1": f"Vencido t-{1+L}", "carteira_growth1": "Cresc. Carteira",
    "selic_meta": "SELIC", "ipca": "IPCA", "desemprego": "Desemprego",
    "pib_var": "PIB mensal (nivel)",
    "spread_pj": "Spread PJ" if A4C else "Saldo cred. SFN",
    "confianca_empresarial": "Conf. Empresarial",
    "credito_pj_saldo": "Saldo Credito PJ",
    "selic_delta3": "SELIC delta 3m",
    "selic_delta6": "SELIC delta 6m",
    "ipca_roll6_mean": "IPCA media 6m",
    "desemprego_delta3": "Desemp. delta 3m",
    "spread_delta3": "Spread PJ delta 3m" if A4C else "Saldo cred. delta 3m",
    "confianca_delta3": "Conf. delta 3m",
    "ibcbr_var12": "IBC-Br var. 12m", "credito_var12": "Cred. SFN var. 12m",
    "cnae_secao_enc": "Setor CNAE", "porte_enc": "Porte",
    "modalidade_enc": "Modalidade",
}
FEATURE_NAMES = [_name_map.get(c, c) for c in FEATURE_COLS]

print("SHAP version:", shap.__version__)
print(f"Features: {len(FEATURE_COLS)}")

def normalizar_shap(shap_vals):
    """Garante array 2D (n_amostras, n_features).
    Para regressão, TreeExplainer retorna 2D diretamente."""
    arr = np.array(shap_vals)
    if arr.ndim == 3:           # (n_amostras, n_features, n_classes)
        return arr[:, :, 1]
    if isinstance(shap_vals, list):
        return np.array(shap_vals[1])
    return arr                  # já 2D


## 1. Carga dos dados e modelos

In [ ]:
df = pd.read_parquet(PROC / "ml_dataset.parquet")
df_test = df[df["data_base"] >= SPLIT_DATE].reset_index(drop=True)

X_test_raw = df_test[FEATURE_COLS]

imputer = SimpleImputer(strategy="median")
df_train = df[df["data_base"] < SPLIT_DATE]
imputer.fit(df_train[FEATURE_COLS])
X_test = pd.DataFrame(imputer.transform(X_test_raw), columns=FEATURE_COLS)
y_test = df_test[TARGET_COL].values

# Carrega regressores
lr  = joblib.load(MDIR / "lr.pkl")
dt  = joblib.load(MDIR / "dt.pkl")
rf  = joblib.load(MDIR / "rf.pkl")
xgb = joblib.load(MDIR / "xgb.pkl")
mlp = joblib.load(MDIR / "mlp.pkl")

# Extrai o regressor de dentro do Pipeline (para SHAP)
lr_reg  = lr[-1]
rf_reg  = rf[-1]
xgb_reg = xgb[-1]

# X escalado para LR/MLP
X_lr  = pd.DataFrame(lr[:-1].transform(X_test),  columns=FEATURE_COLS)
X_mlp = pd.DataFrame(mlp[:-1].transform(X_test), columns=FEATURE_COLS)

np.random.seed(42)
idx = np.random.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
X_shap     = X_test.iloc[idx].values
X_shap_lr  = X_lr.iloc[idx].values
X_shap_xgb = X_test.iloc[idx].values

print(f"Teste: {len(X_test):,} linhas  |  Subsample SHAP: {len(idx):,}")
print("Modelos carregados: LR, DT, RF, XGBoost, MLP")


## 2. SHAP — Random Forest

`TreeExplainer` para regressor baseado em árvore. Valores SHAP indicam o impacto de cada feature na taxa de inadimplência prevista.

In [ ]:
print("Calculando SHAP values — Random Forest...")
explainer_rf = shap.TreeExplainer(rf_reg)
shap_rf = explainer_rf.shap_values(X_shap)
shap_rf = normalizar_shap(shap_rf)

plt.figure()
shap.summary_plot(
    shap_rf, X_shap,
    feature_names=FEATURE_NAMES,
    max_display=20,
    show=False,
    plot_type="dot",
)
plt.title("Random Forest — SHAP Beeswarm (impacto na taxa de inadimplência)")
plt.tight_layout()
plt.savefig(PROC / "shap_rf_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: shap_rf_beeswarm.png")

rf_importance = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "SHAP_mean_abs": np.abs(shap_rf).mean(axis=0),
}).sort_values("SHAP_mean_abs", ascending=False)
print("Top 10 features — Random Forest:")
print(rf_importance.head(10).to_string(index=False))


In [ ]:
# Importância média absoluta
shap.summary_plot(
    shap_rf, X_shap,
    feature_names=FEATURE_NAMES,
    max_display=20,
    show=False,
    plot_type="bar",
)
plt.title("Random Forest — Importância SHAP (média |SHAP|)")
plt.tight_layout()
plt.savefig(PROC / "shap_rf_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: shap_rf_bar.png")

# Tabela de importâncias
rf_importance = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "SHAP_mean_abs": np.abs(shap_rf).mean(axis=0),
}).sort_values("SHAP_mean_abs", ascending=False)
print("\nTop 10 features — Random Forest:")
print(rf_importance.head(10).to_string(index=False))

## 3. SHAP — XGBoost

`TreeExplainer` para o regressor XGBoost. Valores SHAP positivos elevam a previsão de inadimplência; negativos a reduzem.

In [ ]:
print("Calculando SHAP values — XGBoost...")
explainer_xgb = shap.TreeExplainer(xgb_reg)
shap_xgb = explainer_xgb.shap_values(X_shap_xgb)
shap_xgb = normalizar_shap(shap_xgb)

shap.summary_plot(
    shap_xgb, X_shap_xgb,
    feature_names=FEATURE_NAMES,
    max_display=20,
    show=False,
    plot_type="dot",
)
plt.title("XGBoost — SHAP Beeswarm (impacto na taxa de inadimplência)")
plt.tight_layout()
plt.savefig(PROC / "shap_xgb_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

xgb_importance = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "SHAP_mean_abs": np.abs(shap_xgb).mean(axis=0),
}).sort_values("SHAP_mean_abs", ascending=False)
print("Top 10 features — XGBoost:")
print(xgb_importance.head(10).to_string(index=False))


## 4. SHAP — Regressão Linear

`LinearExplainer` para o regressor linear. Os valores SHAP são proporcionais aos coeficientes × valores das features.

In [ ]:
print("Calculando SHAP values — Regressão Linear...")
explainer_lr = shap.LinearExplainer(lr_reg, X_lr.values)
shap_lr = explainer_lr.shap_values(X_shap_lr)
shap_lr = normalizar_shap(shap_lr)

shap.summary_plot(
    shap_lr, X_shap_lr,
    feature_names=FEATURE_NAMES,
    max_display=20,
    show=False,
    plot_type="dot",
)
plt.title("Regressão Linear — SHAP Beeswarm")
plt.tight_layout()
plt.savefig(PROC / "shap_lr_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

lr_importance = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "SHAP_mean_abs": np.abs(shap_lr).mean(axis=0),
}).sort_values("SHAP_mean_abs", ascending=False)
print("Top 10 features — Regressão Linear:")
print(lr_importance.head(10).to_string(index=False))


## 5. Comparação de importâncias entre modelos

Ranking das top 10 features para cada modelo lado a lado — permite identificar consenso e divergências entre as arquiteturas.

In [ ]:
# Importância normalizada (rank 1 = mais importante)
def rank_importance(df_imp):
    df = df_imp.copy().reset_index(drop=True)
    df["rank"] = range(1, len(df) + 1)
    return df.set_index("Feature")["rank"]

rank_rf  = rank_importance(rf_importance)
rank_xgb = rank_importance(xgb_importance)
rank_lr  = rank_importance(lr_importance)

df_rank = pd.DataFrame({
    "RF":      rank_rf,
    "XGBoost": rank_xgb,
    "LR":      rank_lr,
}).fillna(len(FEATURE_NAMES))  # features fora do top recebem rank máximo

# Média dos ranks — consensus importance
df_rank["Média Ranks"] = df_rank.mean(axis=1)
df_rank = df_rank.sort_values("Média Ranks")

print("Ranking de importância por modelo (1 = mais importante):")
print(df_rank.head(15).to_string())

# Gráfico comparativo — top 12 por média
top12 = df_rank.head(12).index.tolist()
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(top12))
width = 0.25

ax.bar(x - width, df_rank.loc[top12, "RF"],      width, label="Random Forest", color="#4daf4a")
ax.bar(x,          df_rank.loc[top12, "XGBoost"], width, label="XGBoost",       color="#984ea3")
ax.bar(x + width,  df_rank.loc[top12, "LR"],      width, label="Reg. Logística", color="#e41a1c")

ax.set_xticks(x)
ax.set_xticklabels(top12, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Rank (menor = mais importante)")
ax.set_title("Importância SHAP — comparação entre modelos (top 12 features)")
ax.invert_yaxis()
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PROC / "shap_comparacao_modelos.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: shap_comparacao_modelos.png")

## 6. SHAP Dependence Plot — feature mais importante

Analisa como o valor de `inadim_lag1` (inadimplência t-1) influencia a predição do XGBoost, com coloração pela segunda feature mais importante.

In [ ]:
# Feature mais importante segundo XGBoost
top_feat_idx = int(np.argmax(np.abs(shap_xgb).mean(axis=0)))
top_feat_name = FEATURE_NAMES[top_feat_idx]
print(f"Feature mais importante (XGBoost): [{top_feat_idx}] {top_feat_name}")

# Segunda mais importante (para coloração)
importance_order = np.argsort(np.abs(shap_xgb).mean(axis=0))[::-1]
second_feat_idx = importance_order[1]

fig, ax = plt.subplots(figsize=(8, 5))
shap.dependence_plot(
    top_feat_idx,
    shap_xgb,
    X_shap_xgb,
    feature_names=FEATURE_NAMES,
    interaction_index=second_feat_idx,
    ax=ax,
    show=False,
)
ax.set_title(f"SHAP Dependence — {top_feat_name} (XGBoost)")
plt.tight_layout()
plt.savefig(PROC / "shap_dependence.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: shap_dependence.png")

## 7. Salvar importâncias consolidadas

In [ ]:
df_importancias = pd.DataFrame({
    "Feature":      FEATURE_NAMES,
    "SHAP_RF":      np.abs(shap_rf).mean(axis=0),
    "SHAP_XGBoost": np.abs(shap_xgb).mean(axis=0),
    "SHAP_LR":      np.abs(shap_lr).mean(axis=0),
})
df_importancias["SHAP_media"] = df_importancias[["SHAP_RF","SHAP_XGBoost","SHAP_LR"]].mean(axis=1)
df_importancias = df_importancias.sort_values("SHAP_media", ascending=False)

df_importancias.to_csv(PROC / "shap_importancias.csv", index=False)
print("Salvo: shap_importancias.csv")
print("\nTop 10 features (média SHAP entre modelos):")
print(df_importancias[["Feature","SHAP_RF","SHAP_XGBoost","SHAP_LR","SHAP_media"]].head(10).to_string(index=False))
print("\nPróximo passo: 06_horizonte.ipynb  (degradação do RMSE vs horizonte t+1..t+12)")

## 8. Figura consolidada — as 12 variáveis mais influentes

Os gráficos `summary_plot` do SHAP exibem 20 variáveis com fonte reduzida, o que os torna
ilegíveis quando reduzidos para pôster ou slide. A figura abaixo apresenta as 12 principais em
formato adequado para apresentação.

In [ ]:
top = df_importancias.head(12).iloc[::-1]        # menor -> maior, para o barh

AZUL, SURF, GRID, AXIS = "#2a78d6", "#fcfcfb", "#e1e0d9", "#c3c2b7"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"

fig, ax = plt.subplots(figsize=(9, 6), facecolor=SURF)
ax.set_facecolor(SURF)
ax.barh(top["Feature"], top["SHAP_media"], color=AZUL, height=0.62)

for y, v in enumerate(top["SHAP_media"]):
    ax.text(v + top["SHAP_media"].max() * 0.015, y, f"{v:.4f}",
            va="center", fontsize=10, color=INK2)

ax.set_xlabel("Importância SHAP média entre modelos  (|SHAP| médio)",
              fontsize=11, color=INK2)
ax.set_title("Variáveis mais influentes na previsão da inadimplência (t+6)",
             fontsize=13, color=INK, pad=12)
ax.set_xlim(0, top["SHAP_media"].max() * 1.16)
ax.tick_params(axis="y", labelsize=11, colors=INK)
ax.tick_params(axis="x", labelsize=10, colors=MUTED)
ax.grid(True, axis="x", color=GRID, linewidth=0.7)
ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
for s in ["left", "bottom"]:
    ax.spines[s].set_color(AXIS)

plt.tight_layout()
plt.savefig(PROC / "shap_top12.png", dpi=200, bbox_inches="tight", facecolor=SURF)
plt.show()
print("Salvo: shap_top12.png  (figura legivel para slides e poster)")